<a href="https://colab.research.google.com/github/AhmedE16/flyrank-ai-intern-ahmed-essam/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [1]:
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb"], check=True)

import duckdb, pandas as pd, numpy as np
from google.colab import userdata

con = duckdb.connect()
hf_token = userdata.get("HF_TOKEN")
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")
REL = "hf://datasets/FlyRank/internship-warehouse"

page_month = con.sql(f"""
    SELECT
        content_hash_id,
        client_hash_id,
        SUM(gsc_impressions) AS impressions_month,
        SUM(gsc_clicks) AS clicks_month,
        AVG(gsc_avg_position) AS avg_position,
        SUM(ga4_sessions) AS sessions_month
    FROM read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')
    WHERE month = '2026-03'
    GROUP BY content_hash_id, client_hash_id
    HAVING SUM(gsc_impressions) > 0
""").df()

page_month["ctr"] = page_month["clicks_month"] / page_month["impressions_month"]
page_month["position_tier"] = pd.cut(
    page_month["avg_position"], bins=[0, 3, 10, 20, 1000],
    labels=["1-3", "4-10", "11-20", "21+"]
)

# Only define the label where a page has enough traffic for CTR to be meaningful --
# otherwise a tier's median CTR gets pinned at 0 by the low-volume tail, and the
# label collapses to all-zero.
MIN_IMPRESSIONS = 100
eligible = page_month["impressions_month"] >= MIN_IMPRESSIONS
page_month = page_month[eligible].reset_index(drop=True)

tier_p25_ctr = page_month.groupby("position_tier", observed=True)["ctr"].transform(lambda s: s.quantile(0.25))
page_month["is_low_ctr"] = (page_month["ctr"] < tier_p25_ctr).astype(int)

print(f"Rows: {len(page_month):,} pages, {page_month['client_hash_id'].nunique()} clients")
print(f"Low-CTR rate: {page_month['is_low_ctr'].mean():.1%}\n")

print("""
METHOD CHOICE:

I'm comparing Logistic Regression and Random Forest against my Week-4 baseline rule.
Logistic Regression as a simple, readable starting point (coefficients can be read
directly), Random Forest because it can capture nonlinear interactions between
position, impressions, and sessions the way Notebook 01/02 already showed beating a
hand-written rule on the starter data. I'm not reaching for Gradient Boosting yet --
this is a CTR-ranking problem with a handful of numeric features, not a case with
enough complexity or data volume to clearly need it, and the assignment says
complexity shouldn't be rewarded on its own.
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 101,441 pages, 44 clients
Low-CTR rate: 2.2%


METHOD CHOICE:

I'm comparing Logistic Regression and Random Forest against my Week-4 baseline rule.
Logistic Regression as a simple, readable starting point (coefficients can be read
directly), Random Forest because it can capture nonlinear interactions between
position, impressions, and sessions the way Notebook 01/02 already showed beating a
hand-written rule on the starter data. I'm not reaching for Gradient Boosting yet --
this is a CTR-ranking problem with a handful of numeric features, not a case with
enough complexity or data volume to clearly need it, and the assignment says
complexity shouldn't be rewarded on its own.



## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [2]:
from sklearn.model_selection import GroupShuffleSplit

# NOTE: clicks_month and impressions_month are deliberately excluded as features --
# ctr = clicks_month / impressions_month, and is_low_ctr is a direct threshold on ctr,
# so including either would let the model reconstruct the label algebraically instead
# of learning a real pattern (this is exactly what happened on the first pass).
features = ["avg_position", "sessions_month"]
X = page_month[features].fillna(0)
y = page_month["is_low_ctr"]
groups = page_month["client_hash_id"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
test_pages = page_month.iloc[test_idx].reset_index(drop=True)

print(f"Train: {len(X_train):,} pages from {groups.iloc[train_idx].nunique()} clients")
print(f"Test:  {len(X_test):,} pages from {groups.iloc[test_idx].nunique()} clients")
print(f"Train label balance: {y_train.mean():.1%} low-CTR")
print(f"Test label balance:  {y_test.mean():.1%} low-CTR")
overlap = set(groups.iloc[train_idx]) & set(groups.iloc[test_idx])
print(f"Clients appearing in BOTH train and test: {len(overlap)} (should be 0)\n")

print("""
SPLIT DESIGN: client-grouped split (GroupShuffleSplit), not a plain random split.

Pages from the same client likely share patterns -- site structure, industry,
content style -- so a random row-level split could let the model "cheat" by
learning client-specific quirks in train and then seeing near-identical pages
from the same client in test. Grouping by client_hash_id keeps every client
entirely in either train or test, so the reported score reflects whether the
model generalizes to genuinely unseen clients, not just unseen rows.
""")

Train: 93,022 pages from 30 clients
Test:  8,419 pages from 14 clients
Train label balance: 2.3% low-CTR
Test label balance:  1.0% low-CTR
Clients appearing in BOTH train and test: 0 (should be 0)


SPLIT DESIGN: client-grouped split (GroupShuffleSplit), not a plain random split.

Pages from the same client likely share patterns -- site structure, industry,
content style -- so a random row-level split could let the model "cheat" by
learning client-specific quirks in train and then seeing near-identical pages
from the same client in test. Grouping by client_hash_id keeps every client
entirely in either train or test, so the reported score reflects whether the
model generalizes to genuinely unseen clients, not just unseen rows.



## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

def precision_at_k(scores, y_true, k):
    order = np.argsort(-np.asarray(scores))[:k]
    return np.asarray(y_true)[order].mean()

# Baseline, reproduced in the same spirit as w04: expected CTR per tier (from TRAIN only, to be fair)
train_tier_ctr = page_month.iloc[train_idx].groupby("position_tier", observed=True)["ctr"].mean().to_dict()
test_pages["expected_ctr"] = test_pages["position_tier"].map(train_tier_ctr).astype(float)
test_pages["ctr_gap"] = test_pages["expected_ctr"] - test_pages["ctr"]
baseline_score = np.where(test_pages["ctr"] < test_pages["expected_ctr"],
                          test_pages["ctr_gap"] * test_pages["impressions_month"], 0)

logreg = LogisticRegression(max_iter=1000, class_weight="balanced")
logreg.fit(X_train, y_train)
logreg_score = logreg.predict_proba(X_test)[:, 1]

rf = RandomForestClassifier(n_estimators=200, max_depth=6, class_weight="balanced", random_state=42)
rf.fit(X_train, y_train)
rf_score = rf.predict_proba(X_test)[:, 1]

comparison = pd.DataFrame({
    "method": ["baseline (w04 rule)", "logistic_regression", "random_forest"],
    "precision_at_20": [
        precision_at_k(baseline_score, y_test, 20),
        precision_at_k(logreg_score, y_test, 20),
        precision_at_k(rf_score, y_test, 20),
    ],
    "precision_at_50": [
        precision_at_k(baseline_score, y_test, 50),
        precision_at_k(logreg_score, y_test, 50),
        precision_at_k(rf_score, y_test, 50),
    ],
})
print(comparison.round(3))

                method  precision_at_20  precision_at_50
0  baseline (w04 rule)             0.05             0.04
1  logistic_regression             0.35             0.28
2        random_forest             0.55             0.42


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [4]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(rf, X_test, y_test, n_repeats=10, random_state=42, scoring="roc_auc")
importance_df = pd.DataFrame({
    "feature": features,
    "importance": perm.importances_mean
}).sort_values("importance", ascending=False)
print("Permutation importance (Random Forest):")
print(importance_df.round(4))

test_pages["rf_score"] = rf_score
test_pages["true_label"] = y_test.values
top50_rf = test_pages.sort_values("rf_score", ascending=False).head(50)
false_positives = top50_rf[top50_rf["true_label"] == 0]

print(f"\nOf the RF's top 50 picks, {len(false_positives)} were false positives (not actually low-CTR).")
print(false_positives[["content_hash_id", "avg_position", "ctr", "impressions_month"]].head(5))

print(f"""
LEAKAGE NOTE: An earlier version of this model included clicks_month and
impressions_month as features. Since ctr = clicks_month / impressions_month and
the label is directly a threshold on ctr, the model was reconstructing the label
algebraically rather than learning a pattern -- this showed up as a suspicious
Precision@20/@50 of 1.00 with zero false positives. Removed those two features
and kept only avg_position and sessions_month, which are associated with CTR but
don't mathematically determine it.

INTERPRETATION: with only avg_position and sessions_month as features, Random
Forest reaches Precision@20 = 0.55 and Precision@50 = 0.42, versus the baseline's
0.05 and 0.04 -- roughly a 10x improvement, and a believable one this time, since
29 of the top 50 picks are false positives rather than zero. avg_position
dominates the permutation importance (0.626 vs 0.022 for sessions_month),
meaning position alone explains most of what the model captures about low CTR --
sessions_month adds only a small amount of additional signal on top of it.

Where the model is wrong: the false positives above tend to be pages at very
strong positions (position ~0.6-1.0, essentially rank #1) that the model still
flagged as high-risk for low CTR, likely because position alone predicts "should
have high CTR" without accounting for query-specific factors like branded or
navigational intent, where even a #1 result can have a modest CTR because users
already know exactly where they're going. This matches the caution from Notebook
01/02: a high score is a candidate for review, not proof of an actual problem.
""")

Permutation importance (Random Forest):
          feature  importance
0    avg_position      0.6263
1  sessions_month      0.0216

Of the RF's top 50 picks, 29 were false positives (not actually low-CTR).
               content_hash_id  avg_position       ctr  impressions_month
5653  content_f3ef589bb0a911eb      1.039787  0.001678              596.0
1467  content_ccb364f88a8824fe      0.593280  0.000889             2249.0
1478  content_2c77cf7d7c787953      0.691639  0.010152              197.0
3750  content_9d8000efca0b24b4      0.732779  0.014388              139.0
5635  content_a494462e8da37fae      0.826338  0.002222              450.0

LEAKAGE NOTE: An earlier version of this model included clicks_month and
impressions_month as features. Since ctr = clicks_month / impressions_month and
the label is directly a threshold on ctr, the model was reconstructing the label
algebraically rather than learning a pattern -- this showed up as a suspicious
Precision@20/@50 of 1.00 with zero fa

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.